<a href="https://colab.research.google.com/github/KhawlahAi/Athkax_program/blob/main/athkax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.precision", 4)

In [2]:
def n_per_group_proportions(p_control, p_treatment, alpha=0.05, power=0.80):
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    var = p_control * (1 - p_control) + p_treatment * (1 - p_treatment)
    return int(np.ceil((z_a + z_b) ** 2 * var / (p_treatment - p_control) ** 2))

def duration_days(n_per_group, daily_new_users, groups=2):
    days = int(np.ceil(n_per_group * groups / daily_new_users))
    return int(np.ceil(days / 7) * 7)

p_control = 0.05
p_treatment = 0.06
daily_new_users = 2000

n_req = n_per_group_proportions(p_control, p_treatment)
days_req = duration_days(n_req, daily_new_users)

print(f"حجم العينة المطلوب لكل مجموعة: {n_req:,}")
print(f"المدة المخططة للتجربة: {days_req} يوماً")

حجم العينة المطلوب لكل مجموعة: 8,155
المدة المخططة للتجربة: 14 يوماً


In [3]:
np.random.seed(2026)
total_users = n_req * 2

# تقسيم المستخدمين عشوائياً 50/50
groups = np.random.choice(['A', 'B'], size=total_users, p=[0.5, 0.5])
data = pd.DataFrame({'user_id': range(1, total_users + 1), 'group': groups})

# محاكاة قرار الاشتراك (نفترض أن B حقق تحسناً حقيقياً بنسبة 6.2%)
prob_A, prob_B = 0.05, 0.062
data['converted'] = np.where(
    data['group'] == 'A',
    np.random.binomial(1, prob_A, total_users),
    np.random.binomial(1, prob_B, total_users)
)

# محاكاة الإيرادات والإلغاء الفوري (فقط لمن اشترك)
revenue_val = np.maximum(0, np.random.normal(100, 20, total_users))
data['revenue'] = data['converted'] * revenue_val

cancel_prob_A, cancel_prob_B = 0.020, 0.021 # نسبة الإلغاء متقاربة جداً
data['cancelled'] = np.where(
    data['converted'] == 1,
    np.where(data['group'] == 'A', np.random.binomial(1, cancel_prob_A, total_users), np.random.binomial(1, cancel_prob_B, total_users)),
    0
)

print("تم توليد بيانات المحاكاة بنجاح. عينة من البيانات:")
display(data.head())

تم توليد بيانات المحاكاة بنجاح. عينة من البيانات:


,user_id,group,converted,revenue,cancelled
0,1,A,0,0.0000,0
1,2,A,0,0.0000,0
2,3,B,0,0.0000,0
3,4,A,1,128.4627,0
4,5,A,0,0.0000,0


In [4]:
counts = data['group'].value_counts()
n_A, n_B = counts['A'], counts['B']
chi2, p_val = stats.chisquare([n_A, n_B], f_exp=[total_users/2, total_users/2])

print(f"المجموعة A: {n_A:,} | المجموعة B: {n_B:,}")
print(f"قيمة p-value لاختبار مربع كاي: {p_val:.4f}")

if p_val < 0.001:
    print("النتيجة: يوجد خلل في التوزيع (SRM)، لا يمكن الوثوق بالنتائج.")
else:
    print("النتيجة: التوزيع سليم.")

المجموعة A: 8,066 | المجموعة B: 8,244
قيمة p-value لاختبار مربع كاي: 0.1634
النتيجة: التوزيع سليم.


In [5]:
conv_A = data[data['group'] == 'A']['converted']
conv_B = data[data['group'] == 'B']['converted']

diff = conv_B.mean() - conv_A.mean()
se = np.sqrt(conv_A.var(ddof=1)/len(conv_A) + conv_B.var(ddof=1)/len(conv_B))
z_score = diff / se
p_val_metric = 2 * stats.norm.sf(abs(z_score))

print(f"معدل التحويل الأساسي (A): {conv_A.mean():.2%}")
print(f"معدل التحويل مع التجربة المجانية (B): {conv_B.mean():.2%}")
print(f"الفرق المطلق: {diff*100:.2f} نقطة مئوية")
print(f"القيمة الاحتمالية (p-value): {p_val_metric:.4f}")

if p_val_metric < 0.05 and diff > 0:
    print("\nالقرار: الأثر إيجابي وذو دلالة إحصائية.")
else:
    print("\nالقرار: لا توجد زيادة ذات دلالة إحصائية.")

معدل التحويل الأساسي (A): 4.86%
معدل التحويل مع التجربة المجانية (B): 6.37%
الفرق المطلق: 1.51 نقطة مئوية
القيمة الاحتمالية (p-value): 0.0000

القرار: الأثر إيجابي وذو دلالة إحصائية.


In [6]:
# فحص الإيراد لكل مستخدم (ARPU)
rev_A = data[data['group'] == 'A']['revenue']
rev_B = data[data['group'] == 'B']['revenue']
diff_rev = rev_B.mean() - rev_A.mean()
se_rev = np.sqrt(rev_A.var(ddof=1)/len(rev_A) + rev_B.var(ddof=1)/len(rev_B))
p_val_rev = 2 * stats.norm.sf(abs(diff_rev / se_rev))

# فحص نسبة الإلغاء
canc_A = data[data['group'] == 'A']['cancelled']
canc_B = data[data['group'] == 'B']['cancelled']
diff_canc = canc_B.mean() - canc_A.mean()
se_canc = np.sqrt(canc_A.var(ddof=1)/len(canc_A) + canc_B.var(ddof=1)/len(canc_B))
p_val_canc = 2 * stats.norm.sf(abs(diff_canc / se_canc))

print(f"الفرق في متوسط الإيراد لكل مستخدم: {diff_rev:.2f} (p-value: {p_val_rev:.4f})")
print(f"الفرق في نسبة الإلغاء الفوري: {diff_canc*100:.2f} نقطة مئوية (p-value: {p_val_canc:.4f})")

if p_val_rev < 0.05 and diff_rev < 0:
    print("تحذير: هناك انخفاض ذو دلالة إحصائية في الإيرادات.")
elif p_val_canc < 0.05 and diff_canc > 0:
    print("تحذير: هناك زيادة ذات دلالة إحصائية في معدل الإلغاء.")
else:
    print("النتيجة: مقاييس الحماية سليمة ولم تتدهور.")

الفرق في متوسط الإيراد لكل مستخدم: 1.60 (p-value: 0.0000)
الفرق في نسبة الإلغاء الفوري: -0.03 نقطة مئوية (p-value: 0.6444)
النتيجة: مقاييس الحماية سليمة ولم تتدهور.
